In [0]:
%sql
CREATE CATALOG IF NOT EXISTS retail_customer

In [0]:
%sql
-- create schemas
USE CATALOG retail_customer;
CREATE SCHEMA IF NOT EXISTS multi_flow1_bronze;
CREATE SCHEMA IF NOT EXISTS multi_flow2_silver;
CREATE SCHEMA IF NOT EXISTS multi_flow3_gold;


In [0]:
%sql
-- Create Volums
CREATE VOLUME IF NOT EXISTS multi_flow1_bronze.bright_home_orders;
CREATE VOLUME IF NOT EXISTS multi_flow1_bronze.lumina_sports_orders;
CREATE VOLUME IF NOT EXISTS multi_flow1_bronze.northstar_outfitters_orders;


In [0]:
# Create path for volum
my_vol_path = '/Volumes/retail_customer/multi_flow1_bronze'


In [0]:
target_folders = ["bright_home_orders", "lumina_sports_orders", "northstar_outfitters_orders"]

for folder in target_folders:
    full_path = f"{my_vol_path}/{folder}"
    print(f"Searching for files in {full_path} to delete prior to creating files...")

    try:
        files = dbutils.fs.ls(full_path)

        if len(files) > 0:
            print(f"--> Found {len(files)} files. Purging path...")
            dbutils.fs.rm(full_path, True)

        else:
            print(f"No files found in {full_path}")
        
    except Exception as e:
        if "FileNotFoundException" in str(e):
            print(f"No files found in {full_path}")
        else:
            print(f"Skipped {folder} due to an unexpected alert: {e}")
    print("-" * 50)


In [0]:
source_volume_path = '/Volumes/databricks_simulated_retail_customer_data/v02/subsidiary_daily_orders'

for folder in target_folders:
    full_path = f"{source_volume_path}/{folder}"
    destination_path = f"{my_vol_path}/{folder}"
    
    print(f"Searching for files in: {full_path}...")
    
    try:
        # 1. List all items in the current directory
        all_items = dbutils.fs.ls(full_path)
        
        # 2. Filter out subdirectories to ensure we only look at files
        files_only = [item for item in all_items if not item.path.endswith('/')]
        
        if files_only:
            # 3. Target the very first file in the list
            first_file_path = files_only[0].path
            first_file_name = files_only[0].name
            
            # Define the exact file target for the destination
            final_destination = f"{destination_path}/{first_file_name}"
            
            print(f"--> Found {len(files_only)} files. Copying the first file: {first_file_name}...")
            
            # 4. Copy the single file (Set recurse=False since it's an individual file)
            dbutils.fs.cp(first_file_path, final_destination, recurse=False)
            print("Done.")
            
        else:
            print(f"Skipped: No actual files found inside {full_path} (folder might be empty or only contain subfolders).")
            
    except Exception as e:
        if "FileNotFoundException" in str(e):
            print(f"Skipped: The source directory {full_path} does not exist.")
        else:
            print(f"Failed to process {folder} due to error: {e}")
            
    print("-" * 50)
